# 03. Policy Pipeline — Authorize every tool call, fail closed

A modern agent runtime spends most of its security budget at one moment: the instant
right before a tool fires. The model has decided what to do. The tool harness has
its arguments ready. Network sockets are warm. **This is the only place where
"don't do that" is still cheap.**

`arctrust.policy` is the engine that lives at that gate. It evaluates every
proposed tool invocation against an ordered stack of policy layers, returns
`allow` or `deny`, and emits a structured audit record. It is short-circuiting
(first DENY wins), fail-closed (an exception in any layer is treated as DENY),
and tier-aware (federal stacks five layers, enterprise four, personal one).

You will learn:

- The shapes that flow through the pipeline: `ToolCall`, `PolicyContext`, `Decision`
- How `PolicyLayer` is just a Protocol — anyone can write one
- How to compose layers into a `PolicyPipeline` and how the five-layer factory works
- Why **first-DENY-wins** is the right default and how to demonstrate it
- Why **fail-closed** is the right default — and how to prove it with an exploding layer
- How `TierConfig` encodes deployment stringency without changing the API surface
- Where this fits in the agent loop (`arcagent`)

This entire notebook runs without an API key, network, filesystem, or clock control.
Tool calls are pure data. Policy decisions are pure functions of those data.

## 1. Setup

The boilerplate cell below makes every `packages/<pkg>/src` and `packages/<pkg>/tests` directory importable. Run it once and forget it for the rest of the notebook.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Pull in every public symbol we need from `arctrust.policy`. Note that the policy engine has no required runtime — no DB, no key file, no network. Just types and a tiny async evaluator.

In [ ]:
from arctrust.policy import (
    Decision,
    PolicyContext,
    PolicyLayer,
    PolicyPipeline,
    TierConfig,
    ToolCall,
    build_pipeline,
)

# Sanity check: these are the five symbols this notebook is responsible for.
public_api = {
    "PolicyContext": PolicyContext,
    "PolicyPipeline": PolicyPipeline,
    "TierConfig": TierConfig,
    "ToolCall": ToolCall,
    "build_pipeline": build_pipeline,
}
for name, obj in public_api.items():
    print(f"{name:18s} -> {obj}")

All five are importable. We also imported `Decision` and `PolicyLayer` because every example below produces or implements them.

## 2. Why a policy pipeline

The agent loop is a long, mostly-deterministic computation that ends each turn
with one or more **tool calls** the model wants to dispatch. Without a policy
gate, the only thing standing between the model and the tool's side effects is
hope.

That fails three ways:

1. **Least privilege.** Most agents only need a tiny slice of the available
   tool surface. A summarizer doesn't need `delete_file`. A search agent
   doesn't need `transfer_funds`. Without an enforced allowlist, "the agent
   could do anything" is functionally true.

2. **Defense in depth.** A model that has been jailbroken or prompt-injected
   does not know it has been compromised. The runtime needs an authority that
   sits *outside* the model's reasoning loop and can refuse to dispatch.

3. **Federal context.** SPEC-017 R-010 to R-018 mandate a layered, fail-closed,
   tamper-evident decision boundary on every tool call. The pipeline IS that
   boundary; everything else (audit, signing, identity) attaches to it.

The pipeline is intentionally **boring**: ordered list of layers, first DENY
wins, exceptions become DENY, every decision is a structured `Decision` object
with the layer / rule / reason / input hash. Boring is the point — federal
auditors don't want clever, they want predictable.

In [ ]:
# A pipeline is just an ordered sequence of layers. Empty pipeline allows
# vacuously — the policy says nothing, so nothing is denied.
empty = PolicyPipeline(layers=[])
print("layers in empty pipeline:", len(empty.layers))

Vacuous-allow is the right default for an *empty* policy. The minute we add even one layer, the pipeline gets opinions.

## 3. Building blocks — `ToolCall`, `PolicyContext`, `TierConfig`

Every evaluation needs three pieces of input:

| Type            | What it carries                                                                          |
|-----------------|------------------------------------------------------------------------------------------|
| `ToolCall`      | The thing the agent wants to do — tool name, arguments, who is asking, what session.     |
| `PolicyContext` | The runtime context — deployment tier, policy bundle version, age of the bundle.         |
| `TierConfig`    | Deployment-tier metadata — which layers run, how many parallel tools allowed.            |

`ToolCall` and `PolicyContext` are Pydantic `frozen=True` models. They are
immutable by construction, which makes them safe to pass around and cache.

In [ ]:
call = ToolCall(
    tool_name="read_file",
    arguments={"path": "/etc/hostname"},
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-001",
    classification="UNCLASSIFIED",
)
print("tool_name:     ", call.tool_name)
print("agent_did:     ", call.agent_did)
print("classification:", call.classification)
print("session_id:    ", call.session_id)
print("parent_call_id:", call.parent_call_id)  # None unless this is a delegated call

`ToolCall` carries an optional `parent_call_id`. When a parent agent delegates to a child, the child's tool calls reference the parent so classification propagation can be audited.

In [ ]:
# Frozen — try to mutate and watch it complain.
try:
    call.tool_name = "delete_file"  # type: ignore[misc]
except Exception as e:
    print(f"frozen model refused mutation: {type(e).__name__}: {e}")

Now the `PolicyContext` — what tier are we, and how stale is the policy bundle? Bundle age matters for restricted mode, where we deny everything except a small safe-set when the bundle hasn't refreshed in a while.

In [ ]:
ctx = PolicyContext(
    tier="personal",
    policy_version="2026-05-07.r1",
    bundle_age_seconds=12.5,
)
print("tier:               ", ctx.tier)
print("policy_version:     ", ctx.policy_version)
print("bundle_age_seconds: ", ctx.bundle_age_seconds)

And `TierConfig` — the metadata that says how stringent we are about evaluating this call. Personal is one layer; enterprise is four; federal is five and caps parallel HTTPS tools at four for FIPS.

In [ ]:
for tier in ("personal", "enterprise", "federal"):
    tc = TierConfig.for_tier(tier)
    print(f"{tier:10s} max_parallel={tc.max_parallel_tools}  layers={tc.layer_names}")

Three things to notice:

- **Personal** runs a single `global` denylist. Lightest touch.
- **Enterprise** adds `provider`, `agent`, and `sandbox` — four layers.
- **Federal** adds `team` between `agent` and `sandbox` — five layers — and the parallel-tool cap drops to 4.

Every deployment evaluates *something*. The difference is depth, not whether the gate exists.

In [ ]:
# An unknown tier name is a programmer error, not a runtime degrade-to-default.
try:
    TierConfig.for_tier("government")  # type: ignore[arg-type]
except ValueError as e:
    print(f"unknown tier rejected: {e}")

## 4. Single-layer evaluation — one rule allowing or denying a call

Before composing layers, build the smallest possible pipeline: one layer, one
rule, one tool call.

`PolicyLayer` is a `runtime_checkable` `Protocol`. Anything with a `name`
attribute and an `async evaluate(call, ctx) -> Decision` satisfies it. No
inheritance, no registration — just shape.

In [ ]:
import time


class WriteOnlyDeny:
    """Trivial layer: deny anything that looks like a write."""

    name = "write_only_deny"

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        is_write = call.tool_name.startswith(("write_", "delete_", "send_"))
        now_us = int(time.monotonic() * 1_000_000)
        if is_write:
            return Decision.deny(
                layer=self.name,
                rule_id="no_writes",
                reason=f"Tool {call.tool_name!r} performs a write; this layer is read-only.",
                input_hash="demo",
                evaluated_at_us=now_us,
            )
        return Decision.allow(input_hash="demo", evaluated_at_us=now_us)


# Confirm the Protocol is satisfied.
layer = WriteOnlyDeny()
assert isinstance(layer, PolicyLayer)
print("WriteOnlyDeny satisfies PolicyLayer protocol:", isinstance(layer, PolicyLayer))

Now build a one-layer pipeline and evaluate two calls — a read (allowed) and a write (denied).

In [ ]:
pipeline = PolicyPipeline(layers=[WriteOnlyDeny()])

read_call = ToolCall(
    tool_name="read_file",
    arguments={"path": "/etc/hostname"},
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-001",
    classification="UNCLASSIFIED",
)
write_call = ToolCall(
    tool_name="delete_file",
    arguments={"path": "/etc/passwd"},
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-001",
    classification="UNCLASSIFIED",
)

ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)

read_decision = await pipeline.evaluate(read_call, ctx)
write_decision = await pipeline.evaluate(write_call, ctx)

print("read_file   ->", read_decision.outcome)
print(
    "delete_file ->",
    write_decision.outcome,
    "| layer:",
    write_decision.layer,
    "| rule:",
    write_decision.rule_id,
    "| reason:",
    write_decision.reason,
)

Three things in the deny output that are not optional:

1. **`layer`** — which layer made the decision (`write_only_deny`).
2. **`rule_id`** — which rule inside that layer (`no_writes`).
3. **`reason`** — human-readable explanation suitable for an audit log.

SPEC-017 R-014 calls these the three structured answers an auditor will demand: which layer, which rule, what inputs. Every `Decision.deny(...)` carries them.

In [ ]:
# A Decision is itself frozen. Once made, it cannot be amended.
try:
    write_decision.outcome = "allow"  # type: ignore[misc]
except Exception as e:
    print(f"frozen decision refused mutation: {type(e).__name__}")

## 5. Pipeline composition — `build_pipeline` and the five-layer stack

Hand-rolling layers is fine for tests and bespoke deployments. For the
common case you want `build_pipeline()`, which assembles the right layer
set for a given tier and wires the standard built-ins:

| Tier        | Layer order                                              | Count |
|-------------|----------------------------------------------------------|-------|
| personal    | `global`                                                 | 1     |
| enterprise  | `global` -> `provider` -> `agent` -> `sandbox`           | 4     |
| federal     | `global` -> `provider` -> `agent` -> `team` -> `sandbox` | 5     |

The factory takes optional `global_deny_rules` (tool-name -> reason),
`agent_allowlists` (DID -> set of tools), `cache_ttl_seconds`,
`max_bundle_age_seconds`, `safe_set`, `shadow`, and `audit_sink`.

Layer ordering is deliberate. Cheapest, broadest checks run first
(`global`); narrowest, most-expensive last (`sandbox`). When `global`
denies, we never reach `sandbox` — saving compute and minimizing what an
adversary can probe by enumerating denials.

In [ ]:
personal_pipeline = build_pipeline(tier="personal")
enterprise_pipeline = build_pipeline(tier="enterprise")
federal_pipeline = build_pipeline(tier="federal")

for label, pipeline in [
    ("personal", personal_pipeline),
    ("enterprise", enterprise_pipeline),
    ("federal", federal_pipeline),
]:
    layer_names = [layer.name for layer in pipeline.layers]
    print(f"{label:10s} ({len(layer_names)} layers): {layer_names}")

The pipeline exposes its layers via `.layers` for introspection. This is a *read-only copy* — mutating the returned list will not mutate the pipeline.

In [ ]:
# Build a personal pipeline with a tenant-wide denylist.
pipeline = build_pipeline(
    tier="personal",
    global_deny_rules={
        "rm_rf": "Recursive delete is never allowed in this tenant.",
        "transfer_funds": "Financial primitives require an enterprise tier.",
        "exfiltrate_data": "Data exfiltration tools are categorically forbidden.",
    },
)

ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)


async def show_global_denylist() -> None:
    for tool in ("read_file", "rm_rf", "transfer_funds", "search_web"):
        call = ToolCall(
            tool_name=tool,
            arguments={},
            agent_did="did:arc:demo:summarizer/0123abcd",
            session_id="sess-002",
            classification="UNCLASSIFIED",
        )
        decision = await pipeline.evaluate(call, ctx)
        flag = "BLOCK" if decision.is_deny() else " ok  "
        detail = f"({decision.layer}/{decision.rule_id})" if decision.is_deny() else ""
        print(f"  {flag}  {tool:18s}  {detail}")


await show_global_denylist()

`global` is enough for personal-tier deployments. For enterprise / federal, add per-agent allowlists so that even tools the tenant *broadly* permits can be locked down for a specific agent.

In [ ]:
# Enterprise pipeline: tenant allows everything except `rm_rf`, but the
# summarizer agent specifically may only call read_file and search_web.
ent_pipeline = build_pipeline(
    tier="enterprise",
    global_deny_rules={"rm_rf": "Recursive delete is never allowed."},
    agent_allowlists={
        "did:arc:demo:summarizer/0123abcd": {"read_file", "search_web"},
    },
)

ctx = PolicyContext(tier="enterprise", policy_version="demo", bundle_age_seconds=0.0)


async def show_enterprise_stack() -> None:
    for tool in ("read_file", "write_file", "rm_rf", "search_web"):
        call = ToolCall(
            tool_name=tool,
            arguments={},
            agent_did="did:arc:demo:summarizer/0123abcd",
            session_id="sess-003",
            classification="UNCLASSIFIED",
        )
        decision = await ent_pipeline.evaluate(call, ctx)
        flag = "BLOCK" if decision.is_deny() else " ok  "
        layer = decision.layer or ""
        print(f"  {flag}  {tool:12s}  layer={layer}")


await show_enterprise_stack()

Two layers each had something to say:

- `global` denied `rm_rf` (tenant-wide rule).
- `agent` denied `write_file` (not on the summarizer's allowlist).

`read_file` and `search_web` ran the full stack and emerged allowed. The fact that the agent layer comes *after* global means that a global denylist can't be circumvented by widening an agent's allowlist — global wins first.

## 6. First-DENY-wins — explicit demo

Pipeline ordering matters because the pipeline **short-circuits on the first
DENY**. A later layer that would have allowed the call never even runs. This
is more than an optimization — it ensures every denial is attributable to a
single, specific layer / rule pair, which is what auditors care about.

Build a deliberately conflicting pair of layers — one denies, the next allows
— and watch the pipeline pick DENY without consulting the second layer.

In [ ]:
import time


class TrackingAllow:
    """Tracks whether it was called. Always allows."""

    name = "tracking_allow"

    def __init__(self) -> None:
        self.called = False

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        self.called = True
        return Decision.allow(
            input_hash="demo",
            evaluated_at_us=int(time.monotonic() * 1_000_000),
        )


class TrackingDeny:
    """Tracks whether it was called. Always denies."""

    name = "tracking_deny"

    def __init__(self) -> None:
        self.called = False

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        self.called = True
        return Decision.deny(
            layer=self.name,
            rule_id="always_deny",
            reason="Configured to deny everything.",
            input_hash="demo",
            evaluated_at_us=int(time.monotonic() * 1_000_000),
        )


early_deny = TrackingDeny()
late_allow = TrackingAllow()

pipeline = PolicyPipeline(layers=[early_deny, late_allow])

call = ToolCall(
    tool_name="any_tool",
    arguments={},
    agent_did="did:arc:demo:any/0123abcd",
    session_id="sess-006",
    classification="UNCLASSIFIED",
)
ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)

decision = await pipeline.evaluate(call, ctx)
print("outcome:                  ", decision.outcome)
print("layer that decided:       ", decision.layer)
print("first (deny) layer called:", early_deny.called)
print("second (allow) layer called:", late_allow.called, "  <-- short-circuited away")

`late_allow.called == False` is the proof. The pipeline saw the DENY from the first layer and stopped — the allow layer never got a chance to overrule it. **No layer can recover from another layer's DENY.**

In [ ]:
# Flip the order. Now the allow layer runs first, sees nothing wrong,
# and the deny layer fires on the second pass — still DENY.
allow_first = TrackingAllow()
deny_after = TrackingDeny()

pipeline2 = PolicyPipeline(layers=[allow_first, deny_after])
decision2 = await pipeline2.evaluate(call, ctx)
print("outcome:                ", decision2.outcome)
print("allow_first.called:     ", allow_first.called)
print("deny_after.called:      ", deny_after.called)

This shows the pipeline does keep walking past `allow` outcomes — it only short-circuits on `deny`. The mental model: every layer is consulted in order until one says no. If none says no, the result is allow.

## 7. Tier-aware decisions — same call, different tier, different verdict

The same `ToolCall` can yield different decisions depending on the tier the
pipeline was built for. This is how Arc supports the personal / enterprise
/ federal tier matrix without three forks of the codebase.

Watch a call hit each tier in turn.

In [ ]:
async def evaluate_across_tiers(tool_name: str, agent_did: str) -> None:
    call = ToolCall(
        tool_name=tool_name,
        arguments={},
        agent_did=agent_did,
        session_id="sess-007",
        classification="UNCLASSIFIED",
    )
    for tier in ("personal", "enterprise", "federal"):
        pipeline = build_pipeline(
            tier=tier,
            global_deny_rules={"rm_rf": "Tenant-wide block."},
            agent_allowlists={
                "did:arc:demo:summarizer/0123abcd": {"read_file"},
            },
        )
        ctx = PolicyContext(tier=tier, policy_version="demo", bundle_age_seconds=0.0)
        decision = await pipeline.evaluate(call, ctx)
        verdict = decision.outcome.upper()
        layer = decision.layer or "-"
        print(f"  {tier:10s} -> {verdict:5s}  (layer: {layer})")


print("call: write_file from summarizer DID")
await evaluate_across_tiers("write_file", "did:arc:demo:summarizer/0123abcd")

print()
print("call: write_file from a different agent (no allowlist entry)")
await evaluate_across_tiers("write_file", "did:arc:demo:planner/9999ffff")

Three things to notice:

1. **Personal tier** has no agent layer, so the summarizer's restricted allowlist
   does not apply — `write_file` slips through. This is correct: personal
   deployments are meant to be light-touch.

2. **Enterprise and federal tiers** both run the agent layer. The summarizer is
   pinned to `read_file` only, so `write_file` is denied at the `agent` layer.

3. **Different agent, same call.** The `planner` DID has *no* allowlist entry —
   it is unconstrained at the agent layer. So `write_file` is allowed for the
   planner across all three tiers. Allowlist is opt-in: if you don't list an
   agent, that layer doesn't restrict it.

In [ ]:
# TierConfig also captures the FIPS parallelism cap. Federal is the only
# tier with a hard cap below 10 (per SPEC-017 R-025).
for tier in ("personal", "enterprise", "federal"):
    tc = TierConfig.for_tier(tier)
    cap = tc.max_parallel_tools
    suffix = " (FIPS cap)" if tier == "federal" else ""
    print(f"  {tier:10s} max_parallel_tools = {cap}{suffix}")

The `arctrust` package itself does not enforce that cap — the agent dispatch
layer in `arcagent` reads `TierConfig.max_parallel_tools` and uses it to size
its tool-execution semaphore. That separation is intentional: `arctrust` is
the leaf library; everything else reads its config and acts on it.

## 8. Fail-closed semantics — exceptions become DENY

Layers MUST NOT raise. But software has bugs. A poorly-written layer might
hit a `KeyError` because someone changed an upstream schema, or run out of
memory, or just deadlock. SPEC-017 R-012 says: **any exception in a layer is
caught by the pipeline and converted to DENY.** It is never allowed to
propagate.

This is "fail-closed" — when the policy engine cannot determine an answer,
the safe answer is no. The alternative ("fail-open" — allow on failure)
would be catastrophic: any adversary who could cause the pipeline to throw
would have effectively bypassed every policy.

Demonstrate it with a layer that simply raises.

In [ ]:
class ExplodingLayer:
    """Always raises. The pipeline must catch this and DENY."""

    name = "exploder"

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        raise RuntimeError("simulated upstream failure: trust store offline")


pipeline = PolicyPipeline(layers=[ExplodingLayer()])

call = ToolCall(
    tool_name="any_tool",
    arguments={},
    agent_did="did:arc:demo:any/0123abcd",
    session_id="sess-008",
    classification="UNCLASSIFIED",
)
ctx = PolicyContext(tier="personal", policy_version="demo", bundle_age_seconds=0.0)

# This call MUST NOT raise — the pipeline catches the layer's RuntimeError
# and converts it to a structured DENY decision.
decision = await pipeline.evaluate(call, ctx)

print("outcome:  ", decision.outcome)
print("layer:    ", decision.layer)
print("rule_id:  ", decision.rule_id)
print("reason:   ", decision.reason)

The `rule_id` is the well-known sentinel `layer_error`. The `reason` includes
the exception type and message — useful for postmortems, useless for an
attacker (no stack frames, no internals). The `layer` is the offending
layer's name, so an SRE can immediately go look at it.

What about a layer that allows, *then* a later layer that explodes? The
exception still wins — and the call is denied.

In [ ]:
class AlwaysAllow:
    name = "always_allow"

    async def evaluate(self, call: ToolCall, ctx: PolicyContext) -> Decision:
        return Decision.allow(
            input_hash="demo",
            evaluated_at_us=int(time.monotonic() * 1_000_000),
        )


pipeline = PolicyPipeline(layers=[AlwaysAllow(), ExplodingLayer()])
decision = await pipeline.evaluate(call, ctx)

print("outcome:", decision.outcome, "| layer:", decision.layer, "| rule:", decision.rule_id)

Even though the *first* layer allowed the call, the *second* raising means
the pipeline cannot honestly answer "allow" — and so it must answer "deny".
This is non-negotiable: the moment any layer becomes unreliable, the whole
pipeline degrades to deny.

If you ever find yourself thinking "but the call really should have been
allowed because the broken layer wasn't relevant", you have rediscovered the
exact reasoning an attacker would weaponize. The pipeline is right.

## 9. Wiring policy into the agent loop

Where does the pipeline actually get called?

In `arcagent`, every `ArcAgent` is constructed with a `PolicyPipeline` (or
the agent builds one via `build_pipeline()` from its tier config). When the
LLM produces tool-use blocks for a turn, the agent's tool dispatcher does
roughly this:

```python
# Pseudocode — see arcagent.core.agent for the real implementation.
for tool_use in response.tool_calls:
    call = ToolCall(
        tool_name=tool_use.name,
        arguments=tool_use.arguments,
        agent_did=self.identity.did,
        session_id=self.session_id,
        classification=self.classification,
        parent_call_id=parent.id if parent else None,
    )
    decision = await self.policy_pipeline.evaluate(call, self.policy_context)
    if decision.is_deny():
        # Emit audit event, raise ToolVetoedError. The model sees the
        # veto reason and decides whether to retry with a different tool
        # or escalate to a human.
        raise ToolVetoedError(decision)
    result = await self.tool_registry.dispatch(call)
```

Three things to internalize about this wiring:

- **The pipeline is the only authority.** There is no second-chance hook,
  no "can we run this anyway" override. If the pipeline denies, the tool
  doesn't run. Period.
- **The decision becomes part of the audit trail.** `arctrust.audit.emit`
  records every evaluation; sinks fan out to JSONL and the signed chain.
  See notebook 04 for that story.
- **Caching is invisible to the agent.** With `cache_ttl_seconds > 0`,
  identical calls in a short window return the same decision without
  re-running layers. The agent never sees the difference.

The simulated dispatch below mimics what `arcagent` does, end to end.

In [ ]:
class ToolVetoedError(RuntimeError):
    """Raised inside the agent loop when the pipeline denies a tool call."""

    def __init__(self, decision: Decision) -> None:
        super().__init__(f"{decision.layer}/{decision.rule_id}: {decision.reason}")
        self.decision = decision


class FakeAgentDispatcher:
    """A 30-line caricature of arcagent.core.agent.ArcAgent's tool-dispatch path."""

    def __init__(
        self,
        *,
        tier: str,
        agent_did: str,
        session_id: str,
        deny_rules: dict[str, str] | None = None,
        agent_allowlists: dict[str, set[str]] | None = None,
    ) -> None:
        self._pipeline = build_pipeline(
            tier=tier,
            global_deny_rules=deny_rules or {},
            agent_allowlists=agent_allowlists or {},
        )
        self._ctx = PolicyContext(tier=tier, policy_version="demo", bundle_age_seconds=0.0)
        self._agent_did = agent_did
        self._session_id = session_id

    async def dispatch(self, tool_name: str, arguments: dict) -> str:
        call = ToolCall(
            tool_name=tool_name,
            arguments=arguments,
            agent_did=self._agent_did,
            session_id=self._session_id,
            classification="UNCLASSIFIED",
        )
        decision = await self._pipeline.evaluate(call, self._ctx)
        if decision.is_deny():
            raise ToolVetoedError(decision)
        return f"executed {tool_name} with {arguments}"


agent = FakeAgentDispatcher(
    tier="enterprise",
    agent_did="did:arc:demo:summarizer/0123abcd",
    session_id="sess-009",
    deny_rules={"rm_rf": "tenant-wide block"},
    agent_allowlists={"did:arc:demo:summarizer/0123abcd": {"read_file", "search_web"}},
)


async def simulate_turn() -> None:
    requested = [
        ("read_file", {"path": "notes.md"}),
        ("write_file", {"path": "out.md", "data": "..."}),
        ("rm_rf", {"path": "."}),
        ("search_web", {"q": "policy pipelines"}),
    ]
    for tool, args in requested:
        try:
            result = await agent.dispatch(tool, args)
            print(f"  ok      {tool:12s}  {result}")
        except ToolVetoedError as e:
            d = e.decision
            print(f"  vetoed  {tool:12s}  layer={d.layer:8s} rule={d.rule_id}")


await simulate_turn()

Inside one simulated agent turn, the model "asked for" four tools. The
pipeline allowed two and vetoed two — once at the `global` layer (`rm_rf`)
and once at the `agent` layer (`write_file`). The vetoes carry their layer
and rule, so the audit trail is precise and the model can decide what to
do next (retry with a different tool, escalate, give up).

For the full agent-side story see:

- `walkthroughs/arcagent/01-first-agent.ipynb` — agent construction
- `walkthroughs/arcagent/02-tool-integration.ipynb` — tool registry + transports
- `walkthroughs/arcagent/03-policy-and-modules.ipynb` — policy + module bus

## 10. Summary

You built and exercised the `arctrust` policy engine end to end.

**What you saw**

- `ToolCall` and `PolicyContext` are immutable inputs. `Decision` is the
  immutable output. All Pydantic frozen models.
- `PolicyLayer` is just a Protocol — anything with a `name` and an async
  `evaluate` qualifies. Custom layers are trivial.
- `PolicyPipeline` runs layers in order, **first DENY wins**, and is
  **fail-closed** — any exception becomes DENY with `rule_id="layer_error"`.
- `TierConfig.for_tier(...)` returns the layer set and parallelism cap for
  personal / enterprise / federal. Federal caps parallel tools at 4 (FIPS).
- `build_pipeline(...)` is the factory — give it a tier, optional deny rules,
  optional allowlists, and it assembles the standard layer stack.

**Public API exercised in this notebook**

| Symbol            | Where                             |
|-------------------|-----------------------------------|
| `PolicyContext`   | section 3, 4, 5, 6, 7, 8, 9       |
| `PolicyPipeline`  | section 4, 5, 6, 7, 8, 9          |
| `TierConfig`      | section 3, 7                      |
| `ToolCall`        | section 3, 4, 5, 6, 7, 8, 9       |
| `build_pipeline`  | section 5, 7, 9                   |
| `Decision`        | section 4, 5, 6, 7, 8, 9 (output) |
| `PolicyLayer`     | section 4 (Protocol check)        |

**Next**

- `walkthroughs/arctrust/04-audit-sinks.ipynb` — every decision above flows
  into an audit trail. That notebook covers `AuditEvent`, `AuditSink`,
  `JsonlSink`, `SignedChainSink`, and `emit`. The `audit_sink` parameter on
  `PolicyPipeline` is the bridge between the two.
- `walkthroughs/arcagent/03-policy-and-modules.ipynb` — see the policy
  pipeline plugged into a real `ArcAgent` with module-bus integration.